# 02 · DML nas tabelas Delta do `bronze`

Reproduz `INSERT`, `UPDATE` e `DELETE` do trabalho 1, agora sobre a tabela `carro` armazenada no MinIO.

In [1]:
from spark_delta_minio_sqlserver.spark_session import create_session

spark = create_session("02-dml-delta")
spark.sparkContext.setLogLevel("ERROR")

CARRO = "delta.`s3a://bronze/carro`"

versao_inicial = spark.sql(f"DESCRIBE HISTORY {CARRO}").agg({"version": "max"}).first()[0]
spark.sql(f"SELECT * FROM {CARRO} ORDER BY placa").show()

26/10/11 12:45:00 WARN Utils: Your hostname, NF-ENG-ISABELLEFELTRIN resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/11 12:45:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/isafel/spark-delta-minio-sqlserver/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/isafel/.ivy2/cache
The jars for the packages stored in: /home/isafel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
com.microsoft.sqlserver#mssql-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-57ad2255-2662-4a65-9b4c-19526877d297;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central


	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 in central
:: resolution report :: resolve 272ms :: artifacts dl 8ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 from central in [default]
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-------

26/10/11 12:45:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


+-------+---------+-----------+--------+----+-------+----------+
|  placa|   modelo|     chassi|marca_id| ano|    cor|cliente_id|
+-------+---------+-----------+--------+----+-------+----------+
|ALD3834|     CLIO|34574215969|       1|2011| BRANCO|         1|
|CCR8096|    CRETA|88547875547|       2|2020| BRANCO|         1|
|DLA3438|    PUNTO|98823483434|       3|2013|  PRETO|         2|
|EEE1056|ECO SPORT|56753453455|       4|2020|   AZUL|         2|
|FFR1234|    PALIO|32383478747|       3|2020|AMARELO|         3|
|GQY6753|      S10|72004160549|       5|2015|  PRETO|         3|
|IAC8974|   TIGUAN|77130757746|       6|2020|   AZUL|         4|
|JIE0952|   PASSAT|87493270405|       6|2016|  CINZA|         4|
+-------+---------+-----------+--------+----+-------+----------+



## INSERT

In [2]:
spark.sql(f"""
    INSERT INTO {CARRO}
    VALUES ('KLM2468', 'ONIX', '11223344556', 5, 2023, 'PRATA', 1)
""")
spark.sql(f"SELECT * FROM {CARRO} WHERE placa = 'KLM2468'").show()

+-------+------+-----------+--------+----+-----+----------+
|  placa|modelo|     chassi|marca_id| ano|  cor|cliente_id|
+-------+------+-----------+--------+----+-----+----------+
|KLM2468|  ONIX|11223344556|       5|2023|PRATA|         1|
+-------+------+-----------+--------+----+-----+----------+



## UPDATE

In [3]:
spark.sql(f"UPDATE {CARRO} SET cor = 'VERMELHO' WHERE placa = 'EEE1056'")
spark.sql(f"SELECT placa, modelo, cor FROM {CARRO} WHERE placa = 'EEE1056'").show()

+-------+---------+--------+
|  placa|   modelo|     cor|
+-------+---------+--------+
|EEE1056|ECO SPORT|VERMELHO|
+-------+---------+--------+



## DELETE

In [4]:
spark.sql(f"DELETE FROM {CARRO} WHERE placa = 'ALD3834'")
spark.sql(f"SELECT * FROM {CARRO} ORDER BY placa").show()

+-------+---------+-----------+--------+----+--------+----------+
|  placa|   modelo|     chassi|marca_id| ano|     cor|cliente_id|
+-------+---------+-----------+--------+----+--------+----------+
|CCR8096|    CRETA|88547875547|       2|2020|  BRANCO|         1|
|DLA3438|    PUNTO|98823483434|       3|2013|   PRETO|         2|
|EEE1056|ECO SPORT|56753453455|       4|2020|VERMELHO|         2|
|FFR1234|    PALIO|32383478747|       3|2020| AMARELO|         3|
|GQY6753|      S10|72004160549|       5|2015|   PRETO|         3|
|IAC8974|   TIGUAN|77130757746|       6|2020|    AZUL|         4|
|JIE0952|   PASSAT|87493270405|       6|2016|   CINZA|         4|
|KLM2468|     ONIX|11223344556|       5|2023|   PRATA|         1|
+-------+---------+-----------+--------+----+--------+----------+



## Histórico e _time travel_

In [5]:
spark.sql(f"DESCRIBE HISTORY {CARRO}").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

+-------+-------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                                                                              |
+-------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
spark.sql(f"SELECT * FROM {CARRO} VERSION AS OF {versao_inicial} ORDER BY placa").show()

+-------+---------+-----------+--------+----+-------+----------+
|  placa|   modelo|     chassi|marca_id| ano|    cor|cliente_id|
+-------+---------+-----------+--------+----+-------+----------+
|ALD3834|     CLIO|34574215969|       1|2011| BRANCO|         1|
|CCR8096|    CRETA|88547875547|       2|2020| BRANCO|         1|
|DLA3438|    PUNTO|98823483434|       3|2013|  PRETO|         2|
|EEE1056|ECO SPORT|56753453455|       4|2020|   AZUL|         2|
|FFR1234|    PALIO|32383478747|       3|2020|AMARELO|         3|
|GQY6753|      S10|72004160549|       5|2015|  PRETO|         3|
|IAC8974|   TIGUAN|77130757746|       6|2020|   AZUL|         4|
|JIE0952|   PASSAT|87493270405|       6|2016|  CINZA|         4|
+-------+---------+-----------+--------+----+-------+----------+



> Re-executar este notebook sem refazer o `01` acumula versões e duplica o `INSERT` da mesma placa. Para recomeçar, rode o `01` de novo (ele sobrescreve a tabela).